# 🛍️ Customer Segmentation — K-Means Clustering
**Dataset:** Mall Customers (Kaggle)  
**Algorithm:** K-Means Clustering  
**Goal:** Segment customers into groups based on income and spending score — without any labels.

> **Why K-Means?** Unsupervised learning — we have no labels. K-Means finds natural groupings in data, ideal for customer segmentation, recommendation systems, and anomaly detection.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Load Mall Customers dataset from GitHub mirror
url = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/mall_customers.csv"
try:
    df = pd.read_csv(url)
except:
    # Fallback: create synthetic data if URL fails
    np.random.seed(42)
    df = pd.DataFrame({
        'CustomerID': range(1, 201),
        'Genre': np.random.choice(['Male','Female'], 200),
        'Age': np.random.randint(18, 70, 200),
        'Annual Income (k$)': np.random.randint(15, 140, 200),
        'Spending Score (1-100)': np.random.randint(1, 100, 200)
    })
    print("Using synthetic data (URL failed). Results still demonstrate K-Means correctly.")

df.columns = [c.strip() for c in df.columns]
print("Shape:", df.shape)
df.head()


## Step 1 — Explore the Data

In [ ]:
print(df.describe())
print("\nMissing values:", df.isnull().sum().sum())

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(df.iloc[:, -2], bins=20, color='#3498DB', edgecolor='black')
plt.title('Annual Income Distribution')

plt.subplot(1, 2, 2)
plt.hist(df.iloc[:, -1], bins=20, color='#E91E63', edgecolor='black')
plt.title('Spending Score Distribution')

plt.tight_layout()
plt.show()


## Step 2 — Select Features & Scale

In [ ]:
# Use Income and Spending Score for 2D clustering (easy to visualize)
X = df.iloc[:, [-2, -1]].values
feat_names = [df.columns[-2], df.columns[-1]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features used: {feat_names}")
print(f"Shape: {X_scaled.shape}")


## Step 3 — Elbow Method to Find Optimal K

In [ ]:
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(K_range, inertias, 'o-', color='#E74C3C', linewidth=2)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (Within-cluster sum of squares)')
axes[0].set_title('Elbow Method', fontsize=13)
axes[0].axvline(x=5, color='gray', linestyle='--', alpha=0.7, label='Elbow at K=5')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 'o-', color='#2ECC71', linewidth=2)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score (higher = better)', fontsize=13)

plt.tight_layout()
plt.show()

print(f"Best K by Silhouette: {K_range[silhouettes.index(max(silhouettes))]}")


**How to read the Elbow plot:** The "elbow" is where inertia stops dropping sharply. Here it's around **K=5** — meaning 5 clusters is a good balance between simplicity and fit.

**Silhouette Score** measures how well-separated clusters are. Higher = better. Values above 0.5 are generally good.


## Step 4 — Fit K-Means with K=5

In [ ]:
K_BEST = 5
kmeans = KMeans(n_clusters=K_BEST, random_state=42, n_init=10)
kmeans.fit(X_scaled)

df['Cluster'] = kmeans.labels_

print("Cluster Sizes:")
print(df['Cluster'].value_counts().sort_index())


## Step 5 — Visualize the Clusters

In [ ]:
colors = ['#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6']
labels = ['Cluster 1','Cluster 2','Cluster 3','Cluster 4','Cluster 5']

plt.figure(figsize=(9, 6))
for i in range(K_BEST):
    mask = df['Cluster'] == i
    plt.scatter(X[mask, 0], X[mask, 1], s=80, c=colors[i], label=f'Cluster {i+1}', alpha=0.8, edgecolors='k', linewidths=0.4)

# Plot centroids (inverse-transform from scaled)
centers = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centers[:, 0], centers[:, 1], s=200, marker='*', c='black', zorder=5, label='Centroids')

plt.xlabel(feat_names[0])
plt.ylabel(feat_names[1])
plt.title(f'Customer Segments — K-Means (K={K_BEST})', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()


## Step 6 — Interpret the Clusters

In [ ]:
print("Cluster Profiles:")
print("=" * 55)
print(df.groupby('Cluster')[list(df.columns[-3:-1])].mean().round(1).to_string())
print("=" * 55)


## Cluster Interpretation (typical for Mall dataset)

| Cluster | Income | Spending | Customer Type |
|---------|--------|----------|---------------|
| 1 | High | High | 🌟 Target Customers (VIP) |
| 2 | Low | Low | 😐 Careful Spenders |
| 3 | High | Low | 💼 High Income but Conservative |
| 4 | Low | High | ⚠️ Overspenders — credit risk |
| 5 | Medium | Medium | 📊 Average Customers |

> **Business Use:** Marketing teams can target Cluster 1 with premium offers, and Cluster 4 with budget-friendly deals.
